# Notebook 09 — Error Analysis

**EvaliSense: AI-Powered Handwritten Examination Evaluation with ML-Based Grading Error Prediction**

---

## Objective

This notebook performs a detailed analysis of model misclassifications to understand when and why the system fails.

We will:
1. Identify false positives (unnecessary reviews) and false negatives (missed errors)
2. Analyze which feature patterns lead to misclassifications
3. Examine the decision boundary and confidence calibration
4. Compare error patterns across different answer quality levels
5. Quantify the practical impact of errors

**Why this matters:** Understanding model failures guides improvement — whether through more data, better features, or architecture changes.

## Setup

In [ ]:
import os, sys, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import config
from models import GradingRiskModel

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = config.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

print("Setup complete ✓")

## 1. Load Model and Data

In [ ]:
# Load data
syn_path = config.samples_dir / 'synthetic_dataset.jsonl'
records = []
with open(syn_path) as f:
    for line in f:
        records.append(json.loads(line.strip()))

feature_keys = sorted(records[0]['features'].keys())
X = np.array([[r['features'][k] for k in feature_keys] for r in records])
y = np.array([1 if r['is_grading_error'] else 0 for r in records])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Load model
model = GradingRiskModel.load(config.risk_model_path)
y_pred = model.model.predict(X_test)
y_prob = model.model.predict_proba(X_test)[:, 1]

print(f"Model: {model.model_name}")
print(f"Test samples: {len(y_test)}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Non-Error', 'Grading Error'])}")

## 2. Misclassification Breakdown

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("╔══════════════════════════════════════════════════╗")
print("║          MISCLASSIFICATION ANALYSIS              ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  True Negatives  (correct non-error):   {tn:>5}    ║")
print(f"║  True Positives  (correct error):       {tp:>5}    ║")
print(f"║  False Positives (unnecessary review):  {fp:>5}    ║")
print(f"║  False Negatives (MISSED error):        {fn:>5}    ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  FP Rate:  {fp/(fp+tn):.1%}  — extra papers flagged       ║")
print(f"║  FN Rate:  {fn/(fn+tp) if (fn+tp) > 0 else 0:.1%}  — ERRORS MISSED (critical)   ║")
print("╚══════════════════════════════════════════════════╝")

if fn > 0:
    print(f"\n  ⚠️  {fn} grading errors would be missed in production.")
    print(f"     These papers would receive incorrect marks without review.")

## 3. Feature Patterns in Misclassifications

In [ ]:
# Identify misclassified samples
fp_mask = (y_test == 0) & (y_pred == 1)  # False Positive
fn_mask = (y_test == 1) & (y_pred == 0)  # False Negative
correct_mask = y_test == y_pred

key_features = ['ocr_confidence', 'overall_semantic_similarity',
                'evaluation_confidence', 'normalized_score',
                'confidence_score_disagreement', 'score_variance']
key_idx = [feature_keys.index(f) for f in key_features]

print(f"{'Feature':<35} {'Correct':>9} {'FP':>9} {'FN':>9}")
print("─" * 65)

for feat, idx in zip(key_features, key_idx):
    correct_mean = X_test[correct_mask, idx].mean() if correct_mask.any() else 0
    fp_mean = X_test[fp_mask, idx].mean() if fp_mask.any() else 0
    fn_mean = X_test[fn_mask, idx].mean() if fn_mask.any() else 0
    print(f"  {feat:<33} {correct_mean:>9.4f} {fp_mean:>9.4f} {fn_mean:>9.4f}")

In [ ]:
# Visualization: FP vs FN feature distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Feature Distributions by Classification Outcome', fontsize=16, fontweight='bold')

for ax, (feat, idx) in zip(axes.flat, zip(key_features, key_idx)):
    groups = []
    labels = []
    colors_plot = []
    
    if correct_mask.any():
        groups.append(X_test[correct_mask, idx])
        labels.append(f'Correct ({correct_mask.sum()})')
        colors_plot.append('#2ECC71')
    if fp_mask.any():
        groups.append(X_test[fp_mask, idx])
        labels.append(f'FP ({fp_mask.sum()})')
        colors_plot.append('#F39C12')
    if fn_mask.any():
        groups.append(X_test[fn_mask, idx])
        labels.append(f'FN ({fn_mask.sum()})')
        colors_plot.append('#E74C3C')
    
    bp = ax.boxplot(groups, labels=labels, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors_plot):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'error_analysis_features.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR / 'error_analysis_features.png'}")

## 4. Prediction Confidence Analysis

In [ ]:
# Analyze how confident the model was in its mistakes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Prediction Confidence Analysis', fontsize=14, fontweight='bold')

# Overall probability distribution
ax = axes[0]
ax.hist(y_prob[y_test == 0], bins=20, alpha=0.6, color='#4A90D9', label='Non-Error', edgecolor='white')
ax.hist(y_prob[y_test == 1], bins=20, alpha=0.6, color='#E74C3C', label='Error', edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', label='Threshold')
ax.set_xlabel('Predicted Probability (Error)')
ax.set_ylabel('Count')
ax.set_title('Score Distribution by True Class')
ax.legend()

# Confidence of correct vs incorrect
ax = axes[1]
correct_conf = np.maximum(y_prob[correct_mask], 1 - y_prob[correct_mask])
incorrect_conf = np.maximum(y_prob[~correct_mask], 1 - y_prob[~correct_mask]) if (~correct_mask).any() else []
data = [correct_conf]
labels = [f'Correct ({len(correct_conf)})']
if len(incorrect_conf) > 0:
    data.append(incorrect_conf)
    labels.append(f'Incorrect ({len(incorrect_conf)})')
bp = ax.boxplot(data, labels=labels, patch_artist=True)
colors_box = ['#2ECC71', '#E74C3C']
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(colors_box[i])
    patch.set_alpha(0.6)
ax.set_ylabel('Max(P, 1-P)')
ax.set_title('Model Confidence: Correct vs Incorrect')

# Calibration-like plot
ax = axes[2]
sorted_idx = np.argsort(y_prob)
ax.scatter(y_prob[sorted_idx], y_test[sorted_idx], alpha=0.3, s=20, color='#4A90D9')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect calibration')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('True Label')
ax.set_title('Calibration Plot')
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Practical Impact Assessment

In [ ]:
# Simulate practical impact of the model
total_papers = len(y_test)
actual_errors = y_test.sum()
flagged = y_pred.sum()

print("╔══════════════════════════════════════════════════╗")
print("║         PRACTICAL IMPACT ASSESSMENT              ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Total papers:              {total_papers:>5}                ║")
print(f"║  Actual grading errors:     {actual_errors:>5} ({actual_errors/total_papers:.1%})         ║")
print(f"║  Papers flagged for review: {flagged:>5} ({flagged/total_papers:.1%})         ║")
print(f"║  True errors caught:        {tp:>5}/{actual_errors} ({tp/actual_errors:.0%})          ║")
print(f"║  Errors missed:             {fn:>5}/{actual_errors}                   ║")
print(f"║  Unnecessary reviews:       {fp:>5}                   ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  WITHOUT EvaliSense:                             ║")
print(f"║    All {total_papers} papers reviewed = {total_papers} reviews        ║")
print(f"║  WITH EvaliSense:                                ║")
print(f"║    Only {flagged} papers reviewed = {flagged} reviews         ║")
print(f"║    Review workload reduced by {(1 - flagged/total_papers):.0%}                ║")
print("╚══════════════════════════════════════════════════╝")

## Observations

1. **False Negatives (most critical):** Missed errors tend to have moderate feature values — they're borderline cases where the AI score is close but not quite right.

2. **False Positives (less critical):** Unnecessary reviews waste examiner time but don't cause incorrect grades. The cost is manageable.

3. **Confidence Gap:** The model is generally more confident on correct predictions. Misclassified samples cluster near the 0.5 decision boundary.

4. **Improvement Paths:**
   - Lower the threshold from 0.5 to 0.3 to catch more errors (at the cost of more false positives)
   - Add more training data, especially borderline cases
   - Introduce additional features (e.g., answer length anomaly, writing quality score)

## Conclusion

The model effectively reduces examiner workload while catching most grading errors. The primary risk (missed errors) can be mitigated by lowering the decision threshold.

**Next:** [Notebook 10 — Final Evaluation & Reporting](./10_final_evaluation.ipynb)